# Day07 — финализация classification mini-product

## D7-01 — аудит handoff и read-only checkpoint

**Цель:** проверить готовность Day07 по SPEC §12. Входы — существующие Day02–Day05 references, Day06 bundle/manifest, consistency report и demo. Выход — inventory с SHA-256, checked metadata и явными missing paths. Данные настоящие, локальные; подготовки или восстановления нет.

**Краткая теория.** Config описывает алгоритм; fitted Pipeline хранит vocabulary/IDF и веса; cached predictions содержат ответы только на прежние строки. `load_inference_reference` проверяет provenance, схемы, IDs/order/labels, splits, CV choice и final cache. `load_bundle` дополнительно проверяет runtime, preprocessing implementation, metadata и fitted state. Равенство contracts связывает две проверки.

Путь проекта: **data → preprocessing → train/group-aware CV → frozen evaluation → inference**.
Путь новых текстов (только схема, здесь не запускается): raw `(N,)` → `minimal-v1` → fitted TF-IDF sparse `(N, V)` → classifier → predictions `(N,)`. Fit vocabulary/IDF разрешён только на train.

В этом пункте нет fit, transform/predict/score, CV, download, serialization или final evaluation. Чтение cached test outputs не создаёт новых test predictions. Загружается только доверенный локальный joblib: checksum проверяет целостность, но не автора.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src/finnews_sentiment").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/finnews_sentiment").is_dir():
    raise RuntimeError("Run from the project root or notebooks directory")
sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.models.inference_artifact import (
    InferenceBundle, InferenceReference, load_bundle, load_inference_reference,
)
print("Checkpoint: D7-01; local artifacts only; no preparation/model calls")

Checkpoint: D7-01; local artifacts only; no preparation/model calls


### Inventory: обязательные и опциональные входы

`required` означает вход принятого handoff для дальнейших задач Day07. Raw archive/canonical CSV и Day01 EDA outputs здесь optional: read-only loaders используют processed data и проверенную provenance, не запускают подготовку raw. Эти файлы понадобятся для отдельных сценариев воспроизведения. Missing required path останавливает checkpoint без автоматического download/rebuild.

Tracked/ignored ниже зафиксированы по Git inventory при создании checkpoint; существующая policy `.gitignore` не меняется. SHA-256 таблица показывает фактические bytes. Наличие файла и его новый hash сами по себе не доказывают корректность: semantic validators вызываются следующей ячейкой.


In [2]:
REQUIRED_GROUPS: dict[str, list[str]] = {'contracts/runtime': ['configs/dataset.json',
                       'configs/train_baseline.json',
                       'configs/experiments_day04.json',
                       'requirements-day01.txt',
                       'requirements-day03.txt',
                       'requirements-dev.txt'],
 'Day02 processed/provenance': ['data/processed/financial_phrasebank_75agree.csv',
                                'reports/day02/preprocessing.json',
                                'reports/day02/duplicates.json',
                                'reports/day02/provenance.json',
                                'reports/day02/observations.md'],
 'Day03 split/baseline': ['data/processed/day03/splits.csv',
                          'reports/day03/config.json',
                          'reports/day03/metrics.json',
                          'reports/day03/provenance.json',
                          'reports/day03/split_summary.json',
                          'reports/day03/validation_predictions.csv',
                          'reports/day03/observations.md'],
 'Day04 CV/selection': ['reports/day04/cv_folds.csv',
                        'reports/day04/cv_fold_summary.csv',
                        'reports/day04/cv_results.csv',
                        'reports/day04/cv_summary.csv',
                        'reports/day04/effective_config.json',
                        'reports/day04/selection.json',
                        'reports/day04/validation_predictions.csv',
                        'reports/day04/metrics.json',
                        'reports/day04/provenance.json',
                        'reports/day04/observations.md'],
 'Day05 final/error analysis': ['docs/day05_failure_cases.csv',
                                'reports/day05/validation_analysis.csv',
                                'reports/day05/validation_errors.csv',
                                'reports/day05/validation_examples.csv',
                                'reports/day05/validation_confusion_counts.csv',
                                'reports/day05/validation_confusion_counts.png',
                                'reports/day05/validation_confusion_normalized.csv',
                                'reports/day05/validation_confusion_normalized.png',
                                'reports/day05/confusion_pairs.csv',
                                'reports/day05/frozen_protocol.json',
                                'reports/day05/test_predictions.csv',
                                'reports/day05/test_metrics.json',
                                'reports/day05/test_confusion_counts.csv',
                                'reports/day05/test_confusion_counts.png',
                                'reports/day05/test_confusion_normalized.csv',
                                'reports/day05/test_confusion_normalized.png',
                                'reports/day05/final_evaluation.json',
                                'reports/day05/analysis_summary.json',
                                'reports/day05/provenance.json',
                                'reports/day05/observations.md'],
 'Day06 bundle/consistency/demo': ['models/day06/sentiment_bundle.joblib',
                                   'models/day06/manifest.json',
                                   'reports/day06/consistency.json',
                                   'docs/day06_examples.csv',
                                   'reports/day06/demo_predictions.csv']}
OPTIONAL_PATHS: list[str] = ['data/raw/FinancialPhraseBank-v1.0.zip',
 'data/raw/source.json',
 'data/interim/financial_phrasebank_75agree.csv',
 'reports/day01/quality.json',
 'reports/day01/class_distribution.csv',
 'reports/day01/text_lengths.csv',
 'reports/day01/sentiment_distribution.png',
 'reports/day01/text_length_distribution.png',
 'reports/day01/observations.md',
 'reports/day01/environment.json']
TRACKED_INPUTS: set[str] = set(['configs/dataset.json',
 'configs/experiments_day04.json',
 'configs/train_baseline.json',
 'data/raw/FinancialPhraseBank-v1.0.zip',
 'data/raw/source.json',
 'docs/day05_failure_cases.csv',
 'docs/day06_examples.csv',
 'reports/day03/observations.md',
 'reports/day04/observations.md',
 'requirements-day01.txt',
 'requirements-day03.txt',
 'requirements-dev.txt'])

required_names = {name for names in REQUIRED_GROUPS.values() for name in names}
input_names = sorted(required_names | set(OPTIONAL_PATHS))
missing_required = [name for name in sorted(required_names) if not (ROOT / name).is_file()]
missing_optional = [name for name in OPTIONAL_PATHS if not (ROOT / name).is_file()]
print("Missing required:", missing_required)
print("Missing optional:", missing_optional)
if missing_required:
    raise FileNotFoundError("Missing Day07 inputs: " + ", ".join(missing_required))
snapshot_before: dict[str, str] = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
    for name in input_names if (ROOT / name).is_file()
}
inventory_rows: list[dict[str, str | bool]] = []
for name in input_names:
    inventory_rows.append({
        "path": name,
        "required": name in required_names,
        "storage": "tracked" if name in TRACKED_INPUTS else "local/ignored",
        "present": (ROOT / name).is_file(),
        "sha256": snapshot_before.get(name, "missing"),
    })
inventory = pd.DataFrame(inventory_rows)
print("Inventory:", inventory.shape, "required:", len(required_names))
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(inventory)

Missing required: []
Missing optional: []
Inventory: (63, 5) required: 53


,path,required,storage,present,sha256
0,configs/dataset.json,True,tracked,True,fc43038477b9beeb491c505244b7b0a403b8da46f062f61cdc8358126382047d
1,configs/experiments_day04.json,True,tracked,True,d9830dd3805252cc523a2fcb6fc22a28638e2c59d31feec11233cb78d908d18a
2,configs/train_baseline.json,True,tracked,True,8ffeb9230ae0cbffd6c00939694d028370ea5e825eaabf1cbb374cc025de7315
3,data/interim/financial_phrasebank_75agree.csv,False,local/ignored,True,791b78a0be391241bd6ea600fdd0128caee7c0dbe89eed10dfd00de505fb3c2e
4,data/processed/day03/splits.csv,True,local/ignored,True,b51d363cac4c00f18d760aa72d3ca6f5272becced6be45d18e763b4b9e557aaf
5,data/processed/financial_phrasebank_75agree.csv,True,local/ignored,True,39dda898a281dc3294be081917b0681524e73edb60f00913183c67714da0c448
6,data/raw/FinancialPhraseBank-v1.0.zip,False,tracked,True,0e1a06c4900fdae46091d031068601e3773ba067c7cecb5b0da1dcba5ce989a6
7,data/raw/source.json,False,tracked,True,f656fab703cbbf88174a45b0bf11b8bf0e716f32c443c1aa385e850c48d12913
8,docs/day05_failure_cases.csv,True,tracked,True,7c4c6381ea288b34c2ca90764d3a2e970550d34345d7d5712ce5b4e832f0823f
9,docs/day06_examples.csv,True,tracked,True,d99af46c74f82bf0fd999a9ea84e4de66e3ed4b7ddb9dbf93f3f6a7a8a9c363c


In [3]:
reference: InferenceReference = load_inference_reference(ROOT)
bundle: InferenceBundle = load_bundle(ROOT / "models/day06/sentiment_bundle.joblib")
assert bundle.metadata.contract == reference.contract, "Bundle/frozen reference mismatch"
assert bundle.metadata.n_train == len(reference.train)
assert bundle.metadata.n_validation == len(reference.validation)
assert bundle.metadata.n_features == reference.day05_n_features
assert reference.validation_predictions["source_row_id"].tolist() == list(reference.contract.validation_source_row_ids)
assert reference.validation_predictions["true_label_id"].tolist() == reference.validation["label_id"].tolist()
print("Candidate:", reference.contract.candidate_id)
print("Fit policy:", reference.contract.fit_policy, "seed:", reference.contract.random_state)
print("Preprocessing:", reference.contract.preprocessing_policy)
print("Class mapping:", reference.contract.label_mapping)
print("Train / validation / saved predictions:", reference.train.shape, reference.validation.shape, reference.validation_predictions.shape)
print("Recorded fitted vocabulary size:", bundle.metadata.n_features)
print("Validated frozen input/final checksums:", len(reference.contract.input_checksums))
print("Recorded replay:", bundle.metadata.validation_replay, "(not rerun)")
print("Runtime:", reference.contract.runtime_versions)
display(reference.split_summary)

Candidate: linear_svc
Fit policy: day03_train_only seed: 42
Preprocessing: minimal-v1
Class mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Train / validation / saved predictions: (2210, 5) (553, 5) (553, 3)
Recorded fitted vocabulary size: 5000
Validated frozen input/final checksums: 28
Recorded replay: exact_match (not rerun)
Runtime: {'python': '3.14.6', 'pandas': '3.0.6', 'numpy': '2.5.3', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.11.2', 'joblib': '1.6.0', 'threadpoolctl': '3.7.0'}


,rows,groups,negative,neutral,positive
split,,,,,
train,2210,2206,269,1374,567
validation,553,552,67,344,142
test,690,690,84,428,178


### Сохранённое evidence Day06

Ниже проверяются hashes inputs/source из существующего consistency report, его привязка к checked bundle и структура сохранённого demo. `check_inference_consistency` не вызывается: он выполняет predictions и при необходимости пишет report. Recorded parity/standalone остаются свидетельствами Day06, свежий validation replay здесь не проводится. Аналогично сравнение demo CSV не измеряет качество на новых текстах.


In [4]:
consistency = json.loads((ROOT / "reports/day06/consistency.json").read_text(encoding="utf-8"))
manifest = json.loads((ROOT / "models/day06/manifest.json").read_text(encoding="utf-8"))
if not isinstance(consistency, dict) or not isinstance(manifest, dict):
    raise ValueError("Expected JSON objects for consistency and manifest")
if consistency.get("policy") != "day06-consistency-v1":
    raise ValueError("Unexpected consistency policy")
checked_report_hashes = 0
for field in ("input_checksums", "source_checksums"):
    checksums = consistency.get(field)
    if not isinstance(checksums, dict) or not checksums:
        raise ValueError("Missing consistency checksum map: " + field)
    for name, expected in checksums.items():
        if not isinstance(name, str) or not isinstance(expected, str):
            raise ValueError("Invalid consistency checksum entry")
        file_path = (ROOT / name).resolve()
        if not file_path.is_relative_to(ROOT):
            raise ValueError("Consistency path outside project")
        assert hashlib.sha256(file_path.read_bytes()).hexdigest() == expected, name
        checked_report_hashes += 1
assert consistency.get("candidate_id") == reference.contract.candidate_id
assert consistency.get("runtime_versions") == reference.contract.runtime_versions
assert consistency.get("state_before_sha256") == consistency.get("state_after_sha256") == manifest.get("state_sha256")
print("Consistency input/source checksum entries checked:", checked_report_hashes)
print("Validated bundle state SHA-256:", manifest["state_sha256"])
print("Recorded parity:", consistency.get("parity"))
print("Recorded standalone status:", consistency.get("standalone", {}).get("status"))

demo_inputs = pd.read_csv(ROOT / "docs/day06_examples.csv", dtype=str, keep_default_na=False)
demo_outputs = pd.read_csv(ROOT / "reports/day06/demo_predictions.csv", dtype=str, keep_default_na=False)
assert demo_inputs.columns.tolist() == ["example_id", "text"]
assert demo_outputs.columns.tolist() == ["example_id", "text", "text_clean", "pred_label_id", "pred_label"]
pd.testing.assert_frame_equal(demo_inputs, demo_outputs[["example_id", "text"]])
assert len(demo_inputs) == 8
print("Saved demo inputs/outputs aligned:", demo_inputs.shape, demo_outputs.shape, "(no new predictions)")

Consistency input/source checksum entries checked: 56
Validated bundle state SHA-256: 7623ab0d4d555aed3c73944f59f85acb23e141ada2794fa826c9d7231383b7f7
Recorded parity: {'n_train': 2210, 'n_validation': 553, 'pipeline_sha256': '228ab432a817bcfcbd4f0b16c7c854f7c0a394d8ae22bedaa4db0724d431ecb8', 'public_api_sha256': '228ab432a817bcfcbd4f0b16c7c854f7c0a394d8ae22bedaa4db0724d431ecb8', 'reference_sha256': '228ab432a817bcfcbd4f0b16c7c854f7c0a394d8ae22bedaa4db0724d431ecb8', 'status': 'exact_match'}
Recorded standalone status: passed
Saved demo inputs/outputs aligned: (8, 2) (8, 5) (no new predictions)


In [5]:
snapshot_after = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
    for name in input_names if (ROOT / name).is_file()
}
assert snapshot_after == snapshot_before, "Day07 inputs changed during read-only audit"
print("Read-only inventory hashes unchanged:", len(snapshot_after))
print("D7-01 complete: checked reference + bundle; D7-02 and later items remain pending.")

Read-only inventory hashes unchanged: 63
D7-01 complete: checked reference + bundle; D7-02 and later items remain pending.


### Интерпретация

Checked reference содержит train `(2210, 5)`, validation `(553, 5)` и сохранённые validation predictions `(553, 3)`; TF-IDF metadata содержит `V=5000`. Это размеры исходных таблиц и fitted state, а не свежие TF-IDF matrices/predictions. Group counts берутся из проверенного split summary. Train-only policy, порядок source IDs, labels, runtime и contracts согласованы; входные hashes до/после равны.

Недостающие обязательные и опциональные inputs перечислены выше. Проверенные loaders не доказывают качество на live news, тождество утраченных весов Day05 или новый validation replay. Validation/test метрики здесь не сравниваются. `git ls-files` также выявил raw ZIP и `data/raw/source.json` как tracked вопреки описанию SPEC §3; `/data/` в `.gitignore` не исключает ранее tracked files. Это зафиксировано в docs/day07.md, index/данные не меняются. README ещё описывает Day01; SPEC §5 содержит исторические placeholders — это входы D7-02/D7-05, в D7-01 они не исправляются. Summary/ограничения и финальное demo будут отдельными пунктами.

### Самопроверка

1. Почему clone с tracked raw archive ещё не содержит готового processed dataset или model bundle? Что нужно ordinary inference?
2. Чем проверка `exists()`, SHA-256 и semantic validation отличаются друг от друга?
3. Что хранит fitted Pipeline, чего нет в saved predictions?
4. Почему чтение final cache не является новой held-out evaluation?
5. Почему `exact_match` metadata не означает, что replay выполнен в этой ячейке?
6. Что нужно отдельно проверить в D7-06, чтобы заявить запуск по README?


## D7-02 — понятная структура и навигация

**Цель:** связать принятые source modules, day docs, notebooks и artifacts без переноса файлов. Вход — D7-01 inventory; выход — проверенная навигация и фактический Git storage. Находки D7-01 выше сохраняют историю аудита; SPEC §3/§5 и docs/day01.md теперь уточнены по этому аудиту. README и последующие пункты остаются вне этого checkpoint.

**Краткая теория.** `src/finnews_sentiment/` содержит reusable logic: data contracts/preparation, deterministic features, training/CV/evaluation и inference. Notebook — учебный caller: показывает формы и результаты тех же функций. Разные notebooks не должны иметь собственные расходящиеся версии preprocessing. CLI `models.train_model` вызывает `run_baseline`; обычный inference использует fitted bundle без train data или fit.

Переименование module может изменить import path и source checksums, сохранённые в manifest/consistency provenance. Поэтому здесь обновляется описание структуры, а существующие modules/entrypoints/артефакты сохраняются. Следующая ячейка только находит source location; import locator не запускает CLI или модель.


In [6]:
from importlib.util import find_spec

navigation_source_hashes: dict[str, str] = {
    str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted((ROOT / "src/finnews_sentiment").rglob("*.py"))
}
navigation_rows: list[dict[str, str]] = [
    {"day": "01", "doc": "docs/day01.md", "notebook": "notebooks/day01_eda.ipynb", "module": "data.prepare_phrasebank"},
    {"day": "02", "doc": "docs/day02.md", "notebook": "notebooks/day02_preprocessing.ipynb", "module": "data.prepare_day02"},
    {"day": "03", "doc": "docs/day03.md", "notebook": "notebooks/day03_baseline.ipynb", "module": "models.train_model"},
    {"day": "04", "doc": "docs/day04.md", "notebook": "notebooks/day04_experiments.ipynb", "module": "models.run_experiments_day04"},
    {"day": "05", "doc": "docs/day05.md", "notebook": "notebooks/day05_error_analysis.ipynb", "module": "models.run_error_analysis_day05"},
    {"day": "06", "doc": "docs/day06.md", "notebook": "notebooks/day06_inference.ipynb", "module": "models.predict"},
    {"day": "07", "doc": "docs/day07.md", "notebook": "notebooks/day07_review.ipynb", "module": "models.inference_artifact"},
]
for row in navigation_rows:
    assert (ROOT / row["doc"]).is_file(), row["doc"]
    assert (ROOT / row["notebook"]).is_file(), row["notebook"]
    module_spec = find_spec("finnews_sentiment." + row["module"])
    if module_spec is None or module_spec.origin is None:
        raise ImportError("Missing module: " + row["module"])
    source_path = ROOT / "src/finnews_sentiment" / (row["module"].replace(".", "/") + ".py")
    assert Path(module_spec.origin).resolve() == source_path.resolve()
    row["source"] = str(source_path.relative_to(ROOT))
display(pd.DataFrame(navigation_rows))
print("Navigation checked: 7 day docs, 7 notebooks, 7 existing module locations; no CLI/model calls.")

,day,doc,notebook,module,source
0,01,docs/day01.md,notebooks/day01_eda.ipynb,data.prepare_phrasebank,src/finnews_sentiment/data/prepare_phrasebank.py
1,02,docs/day02.md,notebooks/day02_preprocessing.ipynb,data.prepare_day02,src/finnews_sentiment/data/prepare_day02.py
2,03,docs/day03.md,notebooks/day03_baseline.ipynb,models.train_model,src/finnews_sentiment/models/train_model.py
3,04,docs/day04.md,notebooks/day04_experiments.ipynb,models.run_experiments_day04,src/finnews_sentiment/models/run_experiments_d...
4,05,docs/day05.md,notebooks/day05_error_analysis.ipynb,models.run_error_analysis_day05,src/finnews_sentiment/models/run_error_analysi...
5,06,docs/day06.md,notebooks/day06_inference.ipynb,models.predict,src/finnews_sentiment/models/predict.py
6,07,docs/day07.md,notebooks/day07_review.ipynb,models.inference_artifact,src/finnews_sentiment/models/inference_artifac...


Navigation checked: 7 day docs, 7 notebooks, 7 existing module locations; no CLI/model calls.


### Git storage и получение generated outputs

`.gitignore` не удаляет уже tracked files из index. В текущем checkout raw ZIP/source.json tracked; canonical/processed CSV, bundle/manifest и final cache local/ignored. Две `reports/day03|day04/observations.md` — tracked exceptions. Таблица ниже проверяет это через Git и показывает, какой stage создаёт outputs. Полная таблица с prerequisites и ссылками на инструкции — docs/day07.md.

Completed final cache берётся из принятого frozen snapshot; read-only loader не запускает новый test при missing/partial artifacts. D7-02 не исполняет ни одну команду подготовки. Ordinary inference может использовать отдельно переданную trusted model pair; dataset и reports для load/predict не требуются.


In [7]:
import subprocess

storage_rows: list[dict[str, str]] = [
    {"path": "data/raw/FinancialPhraseBank-v1.0.zip", "expected": "tracked", "origin": "Pinned source; Day01 check/conversion"},
    {"path": "data/raw/source.json", "expected": "tracked", "origin": "Source metadata; Day01"},
    {"path": "data/interim/financial_phrasebank_75agree.csv", "expected": "local/ignored", "origin": "data.prepare_phrasebank; docs/day01.md"},
    {"path": "data/processed/financial_phrasebank_75agree.csv", "expected": "local/ignored", "origin": "data.prepare_day02; docs/day02.md"},
    {"path": "data/processed/day03/splits.csv", "expected": "local/ignored", "origin": "models.train_model; docs/day03.md"},
    {"path": "reports/day03/observations.md", "expected": "tracked", "origin": "Accepted baseline interpretation"},
    {"path": "reports/day04/observations.md", "expected": "tracked", "origin": "Accepted CV/selection interpretation"},
    {"path": "reports/day05/final_evaluation.json", "expected": "local/ignored", "origin": "Accepted frozen snapshot; read-only cache, docs/day05.md"},
    {"path": "models/day06/sentiment_bundle.joblib", "expected": "local/ignored", "origin": "Trusted pair or explicit frozen build; docs/day06.md"},
    {"path": "models/day06/manifest.json", "expected": "local/ignored", "origin": "Same pair/build; docs/day06.md"},
    {"path": "reports/day06/consistency.json", "expected": "local/ignored", "origin": "Separate consistency runner; docs/day06.md"},
    {"path": "reports/day06/demo_predictions.csv", "expected": "local/ignored", "origin": "CSV inference on docs/day06_examples.csv; docs/day06.md"},
]
tracked_navigation_paths = set(subprocess.check_output(
    ["git", "-C", str(ROOT), "ls-files", "-z"], text=True,
).split("\0"))
ignore_result = subprocess.run(
    ["git", "-C", str(ROOT), "check-ignore", "--stdin"],
    input="\n".join(row["path"] for row in storage_rows) + "\n",
    capture_output=True, text=True, check=False,
)
if ignore_result.returncode not in (0, 1):
    raise RuntimeError(ignore_result.stderr)
ignored_navigation_paths = set(ignore_result.stdout.splitlines())
for row in storage_rows:
    assert (ROOT / row["path"]).is_file(), row["path"]
    storage = "tracked" if row["path"] in tracked_navigation_paths else "local/ignored"
    assert storage == row["expected"], row["path"]
    if storage == "local/ignored":
        assert row["path"] in ignored_navigation_paths, row["path"]
    row["actual"] = storage
display(pd.DataFrame(storage_rows))
assert navigation_source_hashes == {
    str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted((ROOT / "src/finnews_sentiment").rglob("*.py"))
}
print("Git storage checked:", len(storage_rows), "paths; source files unchanged:", len(navigation_source_hashes))

,path,expected,origin,actual
0,data/raw/FinancialPhraseBank-v1.0.zip,tracked,Pinned source; Day01 check/conversion,tracked
1,data/raw/source.json,tracked,Source metadata; Day01,tracked
2,data/interim/financial_phrasebank_75agree.csv,local/ignored,data.prepare_phrasebank; docs/day01.md,local/ignored
3,data/processed/financial_phrasebank_75agree.csv,local/ignored,data.prepare_day02; docs/day02.md,local/ignored
4,data/processed/day03/splits.csv,local/ignored,models.train_model; docs/day03.md,local/ignored
5,reports/day03/observations.md,tracked,Accepted baseline interpretation,tracked
6,reports/day04/observations.md,tracked,Accepted CV/selection interpretation,tracked
7,reports/day05/final_evaluation.json,local/ignored,"Accepted frozen snapshot; read-only cache, doc...",local/ignored
8,models/day06/sentiment_bundle.joblib,local/ignored,Trusted pair or explicit frozen build; docs/da...,local/ignored
9,models/day06/manifest.json,local/ignored,Same pair/build; docs/day06.md,local/ignored


Git storage checked: 12 paths; source files unchanged: 26


### Интерпретация и самопроверка

Все семь day docs/notebooks и исходные import locations найдены. Фактический tracking двенадцати representative paths соответствует карте; source hashes всех 26 Python modules неизменны. Наличие current ignored output не означает, что он доступен после clone. В SPEC §5 описана уже реализованная структура; подробные ML contracts прошлых дней остаются историей принятых этапов. Изменения структуры состоят в документации и навигации; shared preprocessing и estimator state не меняются.

1. Почему `.gitignore` не сделал raw ZIP untracked?
2. Где находится reusable preprocessing и почему его не следует копировать в notebook?
3. Чем `train_model` как CLI отличается от `run_baseline` как runner?
4. Почему простой перенос module может нарушить проверку provenance?
5. Какие inputs нужны ordinary inference, а какие — explicit frozen build?
6. Почему карта с путями и инструкциями получения outputs ещё не доказывает полный запуск README?

Следующий пункт — D7-03: сводка результатов, errors и limitations. Здесь метрики заново не анализируются и новое demo не запускается.


## D7-03 — результаты, error analysis и ограничения

**Цель:** собрать проверенную сводку существующих результатов для [docs/day07_results.md](../docs/day07_results.md). Входы — Day03 baseline/Dummy, Day04 CV/selection/validation и Day05 completed cache/error annotations. Выход — отдельные таблицы по протоколам, interpretation и проверяемые fingerprints. Никаких новых fit, transform/predict/score, CV, download или final run нет.

**Краткая теория.** CV выбирает candidate только внутри прежнего train. Validation подтверждает уже выбранный вариант на тех же 553 строках, где измерен baseline. Held-out test — отдельные 690 строк, открытые один раз в D5-04. Мы читаем checked cache и пересчитываем метрики из сохранённых labels, а не обращаемся к estimator. Baseline/Dummy на test не оценивались и ради таблицы не запускаются.

Macro F1 одинаково взвешивает три class F1; accuracy взвешивает строки и поэтому сильнее зависит от частого neutral. CV mean/std и validation/test scores имеют разные denominators/protocols. Std трёх folds не confidence interval; разница validation/test не paired improvement.


In [8]:
from dataclasses import asdict
from finnews_sentiment.data.experiment_splits import Day04Inputs, load_day04_inputs
from finnews_sentiment.models.error_analysis import ValidationAnalysis, load_validation_analysis
from finnews_sentiment.models.confusion import ValidationConfusion, build_validation_confusion
from finnews_sentiment.models.evaluate import ClassificationMetrics, evaluate_predictions
from finnews_sentiment.models.failure_cases import (
    FAILURE_CATEGORIES, ValidationExamples, select_validation_examples, load_failure_annotations,
)
from finnews_sentiment.models.final_evaluation import FinalEvaluation, load_final_evaluation

D703_SOURCES: tuple[str, ...] = (
    "configs/dataset.json", "configs/train_baseline.json", "configs/experiments_day04.json",
    "data/processed/day03/splits.csv", "reports/day03/provenance.json",
    "reports/day03/split_summary.json", "reports/day03/metrics.json",
    "reports/day03/validation_predictions.csv", "reports/day04/provenance.json",
    "reports/day04/cv_summary.csv", "reports/day04/cv_fold_summary.csv",
    "reports/day04/selection.json", "reports/day04/metrics.json",
    "reports/day04/validation_predictions.csv", "reports/day05/provenance.json",
    "reports/day05/analysis_summary.json", "reports/day05/validation_confusion_counts.csv",
    "reports/day05/validation_confusion_normalized.csv", "reports/day05/confusion_pairs.csv",
    "reports/day05/validation_examples.csv", "docs/day05_failure_cases.csv",
    "reports/day05/frozen_protocol.json", "reports/day05/final_evaluation.json",
    "reports/day05/test_metrics.json", "reports/day06/consistency.json", "models/day06/manifest.json",
)
d703_before: dict[str, str] = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in D703_SOURCES
}
report_provenance = json.loads((ROOT / "reports/day05/provenance.json").read_text())
if not isinstance(report_provenance, dict):
    raise ValueError("Expected Day05 provenance object")
d703_report_hashes = 0
for field in ("input_checksums", "output_checksums"):
    report_checksums = report_provenance.get(field)
    if not isinstance(report_checksums, dict) or not report_checksums:
        raise ValueError("Missing Day05 checksum map: " + field)
    for name, expected in report_checksums.items():
        if not isinstance(name, str) or not isinstance(expected, str):
            raise ValueError("Invalid Day05 checksum entry")
        relative_name = name if field == "input_checksums" else "reports/day05/" + name
        checked_path = (ROOT / relative_name).resolve()
        if not checked_path.is_relative_to(ROOT):
            raise ValueError("Report path outside project")
        assert hashlib.sha256(checked_path.read_bytes()).hexdigest() == expected, relative_name
        d703_report_hashes += 1
print("D7-03 source fingerprints:", len(d703_before))
print("Day05 report input/output checksum entries verified:", d703_report_hashes)


D7-03 source fingerprints: 26
Day05 report input/output checksum entries verified: 39


In [9]:
d703_inputs: Day04Inputs = load_day04_inputs(ROOT)
d703_analysis: ValidationAnalysis = load_validation_analysis(ROOT)
d703_confusion: ValidationConfusion = build_validation_confusion(d703_analysis.table)
d703_final: FinalEvaluation = load_final_evaluation(ROOT)
assert d703_final.cache_status == "reused"
assert d703_final.candidate_id == d703_analysis.candidate_id == reference.contract.candidate_id

saved_baseline_predictions = d703_inputs.validation_predictions
d703_baseline: ClassificationMetrics = evaluate_predictions(
    saved_baseline_predictions["true_label_id"].to_numpy(),
    saved_baseline_predictions["baseline_pred_label_id"].to_numpy(),
)
d703_dummy: ClassificationMetrics = evaluate_predictions(
    saved_baseline_predictions["true_label_id"].to_numpy(),
    saved_baseline_predictions["dummy_pred_label_id"].to_numpy(),
)
assert json.loads(json.dumps(asdict(d703_baseline))) == d703_inputs.metrics["baseline"]
assert json.loads(json.dumps(asdict(d703_dummy))) == d703_inputs.metrics["dummy"]
d703_selected: ClassificationMetrics = d703_confusion.metrics
validation_scores = pd.DataFrame([
    {"model": name, "split": "validation", "n": metrics.n_samples,
     "accuracy": metrics.accuracy, "macro_f1": metrics.macro_f1}
    for name, metrics in (("Dummy most_frequent", d703_dummy), ("LogisticRegression baseline", d703_baseline),
                          (d703_analysis.candidate_id, d703_selected))
])
display(validation_scores)
d703_cv = pd.read_csv(ROOT / "reports/day04/cv_summary.csv", float_precision="round_trip")
display(d703_cv[["candidate_id", "mean_macro_f1", "std_macro_f1", "mean_accuracy"]])
delta_f1: float = d703_selected.macro_f1 - d703_baseline.macro_f1
relative_delta_f1: float = delta_f1 / d703_baseline.macro_f1 * 100
print(f"Validation macro F1 delta: {delta_f1:.12f}; relative: {relative_delta_f1:.12f}%")
print("CV: 2210 train rows / 2206 groups, 3 shared folds, ddof=0; max mean macro F1; exact ties use frozen order.")
print("CV std is descriptive; 6 rows represent 5 unique configurations, 3 improvement directions.")
print("Frozen TEST only:", d703_final.candidate_id, asdict(d703_final.confusion.metrics))
print("Baseline/Dummy TEST: not evaluated; no model calls added.")
print("Shapes: train DataFrame", reference.train.shape, "; recorded fitted TF-IDF (2210, 5000); saved validation predictions (553,); validation confusion", d703_confusion.counts.shape)
display(pd.DataFrame([asdict(item) for item in d703_selected.per_class]))
display(pd.DataFrame([asdict(item) for item in d703_final.confusion.metrics.per_class]))


,model,split,n,accuracy,macro_f1
0,Dummy most_frequent,validation,553,0.622061,0.255667
1,LogisticRegression baseline,validation,553,0.813743,0.720965
2,linear_svc,validation,553,0.848101,0.805961


,candidate_id,mean_macro_f1,std_macro_f1,mean_accuracy
0,baseline,0.664277,0.026227,0.792315
1,word_unigram,0.669960,0.019630,0.789599
2,linear_svc,0.768715,0.024064,0.836660
3,logreg_c_0_1,0.381356,0.023907,0.672852
4,logreg_c_1_0,0.664277,0.026227,0.792315
5,logreg_c_10_0,0.759263,0.023086,0.832588


Validation macro F1 delta: 0.084995483491; relative: 11.789125301489%
CV: 2210 train rows / 2206 groups, 3 shared folds, ddof=0; max mean macro F1; exact ties use frozen order.
CV std is descriptive; 6 rows represent 5 unique configurations, 3 improvement directions.
Frozen TEST only: linear_svc {'n_samples': 690, 'accuracy': 0.8565217391304348, 'macro_f1': 0.797050158244188, 'per_class': ({'class_id': 0, 'class_name': 'negative', 'precision': 0.9411764705882353, 'recall': 0.5714285714285714, 'f1': 0.7111111111111111, 'support': 84}, {'class_id': 1, 'class_name': 'neutral', 'precision': 0.8589211618257261, 'recall': 0.9672897196261683, 'f1': 0.9098901098901099, 'support': 428}, {'class_id': 2, 'class_name': 'positive', 'precision': 0.821656050955414, 'recall': 0.7247191011235955, 'f1': 0.7701492537313432, 'support': 178})}
Baseline/Dummy TEST: not evaluated; no model calls added.
Shapes: train DataFrame (2210, 5) ; recorded fitted TF-IDF (2210, 5000); saved validation predictions (553,

,class_id,class_name,precision,recall,f1,support
0,0,negative,0.870370,0.701493,0.776860,67
1,1,neutral,0.851175,0.947674,0.896836,344
2,2,positive,0.827586,0.676056,0.744186,142


,class_id,class_name,precision,recall,f1,support
0,0,negative,0.941176,0.571429,0.711111,84
1,1,neutral,0.858921,0.967290,0.909890,428
2,2,positive,0.821656,0.724719,0.770149,178


### Validation errors: denominators и наблюдения

Выбор примеров остаётся прежним: до трёх минимальных source_row_id на каждую из шести true→pred пар. Это иллюстративная выборка, а не случайная оценка распространённости причин. Проверяем existing annotations через source IDs, не позицию CSV. Counts учитывают строки, включая retained duplicates; число exact-text groups показывается отдельно. Матрица `(3, 3)`: строки true, столбцы predicted; нормировка по true-class support.

Следующая ячейка повторно строит лишь summaries из saved labels и проверяет их против Day05 CSV/JSON. Сам report runner не вызывается: он мог бы писать outputs. Новые test examples не просматриваются.


In [10]:
d703_examples: ValidationExamples = select_validation_examples(d703_analysis.table)
d703_reviewed = load_failure_annotations(ROOT / "docs/day05_failure_cases.csv", d703_examples.examples)
saved_reviewed = pd.read_csv(ROOT / "reports/day05/validation_examples.csv", keep_default_na=False)
pd.testing.assert_frame_equal(d703_reviewed.reset_index(drop=True), saved_reviewed)
saved_counts = pd.read_csv(ROOT / "reports/day05/validation_confusion_counts.csv", index_col=0)
saved_counts.columns.name = "pred_label"
pd.testing.assert_frame_equal(d703_confusion.counts, saved_counts)
saved_normalized = pd.read_csv(ROOT / "reports/day05/validation_confusion_normalized.csv", index_col=0, float_precision="round_trip")
saved_normalized.columns.name = "pred_label"
pd.testing.assert_frame_equal(d703_confusion.normalized, saved_normalized, check_exact=True)
saved_pairs = pd.read_csv(ROOT / "reports/day05/confusion_pairs.csv", float_precision="round_trip")
pd.testing.assert_frame_equal(d703_confusion.pairs, saved_pairs, check_exact=True)
print("Validation rows/groups:", len(d703_analysis.table), int(d703_analysis.table["group_id"].nunique()))
print("Error rows/groups:", len(d703_analysis.errors), int(d703_analysis.errors["group_id"].nunique()))
print("Reviewed rows/groups:", len(d703_reviewed), int(d703_reviewed["group_id"].nunique()))
display(d703_confusion.counts)
display(d703_confusion.pairs)
category_counts = pd.DataFrame([
    {"category": category, "count": int(d703_reviewed["primary_category"].eq(category).sum()),
     "reviewed_rows": len(d703_reviewed)} for category in FAILURE_CATEGORIES
])
display(category_counts)
display(d703_reviewed.loc[d703_reviewed["source_row_id"].isin([2854, 2598, 1294, 321]),
    ["source_row_id", "true_label", "pred_label", "observation", "hypothesis"]])
saved_analysis_summary = json.loads((ROOT / "reports/day05/analysis_summary.json").read_text())
assert saved_analysis_summary["validation"]["macro_f1"] == d703_selected.macro_f1
assert saved_analysis_summary["validation"]["n_errors"] == len(d703_analysis.errors)
assert saved_analysis_summary["test"]["macro_f1"] == d703_final.confusion.metrics.macro_f1
assert saved_analysis_summary["manual_review"]["source_row_ids"] == d703_reviewed["source_row_id"].tolist()
assert [(row["category"], row["count"]) for row in saved_analysis_summary["manual_review"]["category_counts"]] == list(zip(category_counts["category"], category_counts["count"], strict=True))


Validation rows/groups: 553 552
Error rows/groups: 84 84
Reviewed rows/groups: 18 18


pred_label,negative,neutral,positive
true_label,,,
negative,47,15,5
neutral,3,326,15
positive,4,42,96


,true_label,pred_label,count,true_support,fraction_of_true
0,negative,neutral,15,67,0.223881
1,negative,positive,5,67,0.074627
2,neutral,negative,3,344,0.008721
3,neutral,positive,15,344,0.043605
4,positive,negative,4,142,0.028169
5,positive,neutral,42,142,0.295775


,category,count,reviewed_rows
0,mixed sentiment,0,18
1,financial context,12,18
2,negation/comparison,3,18
3,insufficient context/ambiguous label,2,18
4,preprocessing/representation limitation,0,18
5,other/unclear,1,18


,source_row_id,true_label,pred_label,observation,hypothesis
4,2854,negative,positive,EPS описан как loss of EUR0 .05.,"Возможно, шаблон EPS amounted to перевешивает ..."
8,2598,neutral,negative,Сообщается о снижении доли владения ниже порог...,"Возможно, decreased ассоциируется со снижением..."
12,321,positive,negative,Указан рост net sales на 28% к первому полугод...,
13,1294,positive,negative,Ожидается переход от убытка EUR9 .6 m годом ра...,"Возможно, историческое loss влияет на предсказ..."


### Интерпретация и ограничения

Выбранный по train-only CV LinearSVC имеет validation macro F1 0.805961 против baseline 0.720965: абсолютная разница +0.084995, относительная +11.79%. Это относится к тем же validation IDs; baseline test не измерялся. Frozen test macro F1 0.797050 и accuracy 0.856522 — отдельный сохранённый результат. Neutral 428/690 определяет большую часть accuracy; negative recall 48/84 = 0.571429 показывает важный minority trade-off.

Validation errors — 84/553, в том числе positive→neutral 42/142 строк с true_label=positive и negative→neutral 15/67 строк с true_label=negative. Среди 18 просмотренных примеров financial context 12/18, negation/comparison 3/18, insufficient context/ambiguous label 2/18, other/unclear 1/18. Это не распределение причин среди всех 84 ошибок. IDs 2854/2598 иллюстрируют EPS loss/ownership; ID 1294 — переход от прошлого loss к ожидаемой прибыли; ID 321 — ошибка при явном росте sales с неустановленной причиной. Наблюдения не доказывают механизм, похожие тексты с разными group IDs не доказывают независимость.

Область применимости ограничена английскими финансовыми предложениями Financial PhraseBank sentences_75agree, одним group split и исторической тематикой. Domain/time shift, неоднозначная разметка, OOV и bag-of-ngrams ограничивают перенос. Нельзя обещать понимание контекста/сравнений, probabilities у LinearSVC или качество по unlabeled demo. Тональность не прогноз доходности и не торговая рекомендация. Joblib только trusted-local, переносимость между runtime versions не гарантируется.

Будущие гипотезы из Day05 — обработка временных сравнений/области отрицания и типа финансового события. Они требуют отдельного заранее согласованного train/CV протокола. Здесь новые features, thresholds, selection или tuning не проверяются; уже открытый test не становится независимым holdout для будущих изменений.


In [11]:
d703_after = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in D703_SOURCES
}
assert d703_after == d703_before, "Result sources changed during summary"
d703_payload: dict[str, object] = {
    "verified_on": "2026-10-25", "cache_status": d703_final.cache_status,
    "candidate": d703_analysis.candidate_id,
    "baseline": asdict(d703_baseline), "dummy": asdict(d703_dummy),
    "selected_validation": asdict(d703_selected), "frozen_test": asdict(d703_final.confusion.metrics),
    "cv": d703_cv.to_dict("records"), "delta_absolute": delta_f1, "delta_relative_percent": relative_delta_f1,
    "validation_counts": d703_confusion.counts.to_numpy().tolist(),
    "confusion_pairs": d703_confusion.pairs.to_dict("records"),
    "category_counts": category_counts.to_dict("records"),
    "n_validation_groups": int(d703_analysis.table["group_id"].nunique()),
    "n_errors": len(d703_analysis.errors), "n_error_groups": int(d703_analysis.errors["group_id"].nunique()),
    "n_reviewed": len(d703_reviewed), "n_reviewed_groups": int(d703_reviewed["group_id"].nunique()),
    "reviewed_ids": d703_reviewed["source_row_id"].tolist(),
    "report_checksum_entries": d703_report_hashes, "source_checksums": d703_before,
}
print("D7-03: read-only cache status:", d703_final.cache_status, "; result source hashes unchanged:", len(d703_after))


D7-03: read-only cache status: reused ; result source hashes unchanged: 26


### Самопроверка

1. Почему CV выбирает candidate, а validation лишь подтверждает выбор?
2. Чем абсолютная delta macro F1 отличается от относительных процентов? Почему их считают до округления?
3. Почему разница 0.805961 validation и 0.797050 test не показывает paired ухудшение?
4. Почему accuracy 0.856522 совместима с negative recall 0.571429?
5. Чем 12/18 financial context отличается от доли среди всех 84 ошибок?
6. Что доказывает cached replay и почему нельзя использовать уже открытый test для нового tuning?

Следующий пункт — D7-04: demo Python API/CSV CLI. Этот checkpoint не запускает demo predictions и не меняет README.


## D7-04 — demo Python API и CSV CLI

**Цель:** показать работающий inference на восьми прежних unlabeled `docs/day06_examples.csv`. Входы — trusted fitted bundle/manifest и существующий demo CSV; выход — typed API records и временный CSV, совпадающие с принятым Day06 output. Входные фразы, порядок и модель не меняются ради ответов.

**Краткая теория.** Raw texts `(8,)` → общая `preprocess_text` (`minimal-v1`) → fitted sparse TF-IDF `(8, 5000)` → LinearSVC labels `(8,)`. `predict_texts` принимает batch (для одной строки `[text]`), сохраняет duplicates и порядок, не делает fit и не возвращает probabilities. CSV CLI использует те же функции, сохраняет input fields и добавляет `text_clean`, `pred_label_id`, `pred_label`.

Это smoke demo, не quality measurement: ground-truth labels отсутствуют. Accuracy/F1 по этим восьми строкам не считаются. Контракты ошибок и edge cases проверяются existing offline tests, а не исправлением model outputs. Предыдущие sections показывают исторические метрики отдельно.


In [12]:
from finnews_sentiment.models.predict import SentimentPrediction, predict_texts
from finnews_sentiment.models.inference_artifact import _state_sha256 as inference_state_sha256

d704_input_path = ROOT / "docs/day06_examples.csv"
d704_saved_path = ROOT / "reports/day06/demo_predictions.csv"
d704_model_path = ROOT / "models/day06/sentiment_bundle.joblib"
d704_protected_names: tuple[str, ...] = (
    "docs/day06_examples.csv", "reports/day06/demo_predictions.csv",
    "models/day06/sentiment_bundle.joblib", "models/day06/manifest.json",
)
d704_hashes_before: dict[str, str] = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in d704_protected_names
}
d704_state_before: str = inference_state_sha256(bundle)
assert d704_state_before == json.loads(d704_model_path.with_name("manifest.json").read_text())["state_sha256"]
d704_inputs = pd.read_csv(d704_input_path, dtype=str, keep_default_na=False)
assert d704_inputs.columns.tolist() == ["example_id", "text"]
assert len(d704_inputs) == 8 and d704_inputs["example_id"].is_unique
d704_texts: list[str] = d704_inputs["text"].tolist()
d704_predictions: list[SentimentPrediction] = predict_texts(bundle, d704_texts)
d704_again: list[SentimentPrediction] = predict_texts(bundle, d704_texts)
assert d704_again == d704_predictions
d704_duplicate_probe: list[SentimentPrediction] = predict_texts(
    bundle, [d704_texts[0], d704_texts[0], d704_texts[-1]],
)
assert d704_duplicate_probe == [d704_predictions[0], d704_predictions[0], d704_predictions[-1]]
d704_table = pd.DataFrame([
    {"example_id": example_id, **asdict(prediction)}
    for example_id, prediction in zip(d704_inputs["example_id"], d704_predictions, strict=True)
])
pd.testing.assert_frame_equal(d704_table[["example_id", "text"]], d704_inputs)
print("Mapping:", bundle.metadata.contract.label_mapping)
print("API: 8 predictions; exact repeat; duplicate/order probe 01, 01, 08 preserved.")
with pd.option_context("display.max_colwidth", None):
    display(d704_table)
assert inference_state_sha256(bundle) == d704_state_before


Mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
API: 8 predictions; exact repeat; duplicate/order probe 01, 01, 08 preserved.


,example_id,text,text_clean,pred_label_id,pred_label
0,01,Aster Quay Robotics reported a 17% rise in operating profit to EUR 8.4 million.,aster quay robotics reported a 17% rise in operating profit to eur 8.4 million.,2,positive
1,02,Boreal Lantern Foods posted a net loss of EUR 6.2 million after a warehouse closure.,boreal lantern foods posted a net loss of eur 6.2 million after a warehouse closure.,1,neutral
2,03,Cobalt Orchard Systems will hold its annual general meeting on 19 November.,cobalt orchard systems will hold its annual general meeting on 19 november.,1,neutral
3,04,Delta Pebble Networks did not increase revenue this quarter despite higher sales volumes.,delta pebble networks did not increase revenue this quarter despite higher sales volumes.,2,positive
4,05,Ember Vale Packaging earned less than last year as shipping costs increased by 23%.,ember vale packaging earned less than last year as shipping costs increased by 23%.,2,positive
5,06,<b>Fjord Mosaic Energy</b> signed a contract worth €4.7 million &amp; expects deliveries in 2028.,fjord mosaic energy signed a contract worth €4.7 million & expects deliveries in 2028.,1,neutral
6,07,Garnet Tide Analytics kept its full-year guidance unchanged at USD 31 million in sales.,garnet tide analytics kept its full-year guidance unchanged at usd 31 million in sales.,1,neutral
7,08,zxqvplmnbvcxqz,zxqvplmnbvcxqz,1,neutral


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix

d704_vectorizer = bundle.pipeline.named_steps["tfidf"]
if not isinstance(d704_vectorizer, TfidfVectorizer):
    raise TypeError("Expected fitted TfidfVectorizer")
d704_features = csr_matrix(d704_vectorizer.transform([item.text_clean for item in d704_predictions]))
d704_oov_nnz: int = int(d704_features.getrow(7).nnz)
d704_feature_shape: tuple[int, int] = (len(d704_predictions), bundle.metadata.n_features)
assert d704_features.shape == d704_feature_shape
assert d704_oov_nnz == 0
print("Fitted TF-IDF:", d704_features.shape, "; OOV example 08 nnz:", d704_oov_nnz)
print("06 raw:", d704_predictions[5].text)
print("06 clean:", d704_predictions[5].text_clean)
print("02/04/05 labels:", [d704_predictions[index].pred_label for index in (1, 3, 4)])
assert inference_state_sha256(bundle) == d704_state_before


Fitted TF-IDF: (8, 5000) ; OOV example 08 nnz: 0
06 raw: <b>Fjord Mosaic Energy</b> signed a contract worth €4.7 million &amp; expects deliveries in 2028.
06 clean: fjord mosaic energy signed a contract worth €4.7 million & expects deliveries in 2028.
02/04/05 labels: ['neutral', 'positive', 'positive']


### CSV CLI: новый output path и сверка всех fields

Команда запускается отдельным процессом из корня, использует тот же bundle и input. `--output` должен отсутствовать: existing output не заменяется. Поэтому принятый `reports/day06/demo_predictions.csv` только читается; оба CLI повтора записываются в разные новые пути внутри `TemporaryDirectory`, после сравнения временные файлы удаляются.

Все input columns читаются строками (`dtype=str`, `keep_default_na=False`), поэтому IDs `01`–`08` не превращаются в числа. В ответе `pred_label_id` записан integer literal, при сравнении CSV прочитан как строка. Проверяем bytes принятого output, raw/clean/labels, порядок и parity с Python API. При missing/bad bundle, blank/invalid text, malformed CSV, reserved columns или existing output CLI завершается с error (exit 2), не делает скрытого build/overwrite.


In [14]:
import os
from tempfile import TemporaryDirectory

d704_saved = pd.read_csv(d704_saved_path, dtype=str, keep_default_na=False)
d704_expected_api = d704_table.assign(pred_label_id=[str(item.pred_label_id) for item in d704_predictions])
pd.testing.assert_frame_equal(d704_expected_api, d704_saved)
d704_cli_hashes: list[str] = []
d704_cli_exit_codes: list[int] = []
with TemporaryDirectory(prefix="day07-demo-") as temporary_directory:
    for attempt in range(2):
        d704_cli_path = Path(temporary_directory) / f"predictions-{attempt}.csv"
        d704_environment = dict(os.environ)
        d704_environment["PYTHONPATH"] = str(ROOT / "src")
        d704_command = [
            sys.executable, "-m", "finnews_sentiment.models.predict",
            "--model", str(d704_model_path), "--input", str(d704_input_path),
            "--output", str(d704_cli_path),
        ]
        d704_cli_run = subprocess.run(
            d704_command, cwd=ROOT, env=d704_environment,
            check=True, capture_output=True, text=True,
        )
        d704_cli_exit_codes.append(d704_cli_run.returncode)
        d704_cli_bytes = d704_cli_path.read_bytes()
        assert d704_cli_bytes == d704_saved_path.read_bytes()
        d704_cli_table = pd.read_csv(d704_cli_path, dtype=str, keep_default_na=False)
        pd.testing.assert_frame_equal(d704_cli_table, d704_expected_api)
        d704_cli_hashes.append(hashlib.sha256(d704_cli_bytes).hexdigest())
assert len(set(d704_cli_hashes)) == 1
assert inference_state_sha256(bundle) == d704_state_before
d704_hashes_after = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in d704_protected_names
}
assert d704_hashes_after == d704_hashes_before
d704_payload: dict[str, object] = {
    "n_examples": len(d704_inputs), "predictions": d704_table.to_dict("records"),
    "api_repeat_exact": True, "duplicates_preserved": True,
    "cli_exit_codes": d704_cli_exit_codes, "cli_output_hashes": d704_cli_hashes,
    "api_cli_saved_exact": True, "tfidf_shape": list(d704_feature_shape), "oov_nnz": int(d704_oov_nnz),
    "state_before_sha256": d704_state_before, "state_after_sha256": inference_state_sha256(bundle),
    "file_checksums": d704_hashes_before,
}
print("CLI exit codes:", d704_cli_exit_codes)
print("Two CLI outputs match all saved/API fields and accepted CSV bytes:", d704_cli_hashes[0])
print("Input/saved output/bundle/manifest unchanged; fitted state:", d704_state_before)
print("Temporary CSV files removed; no labels, accuracy/F1 or model tuning.")


CLI exit codes: [0, 0]
Two CLI outputs match all saved/API fields and accepted CSV bytes: 2d689d528e0c4520e5b9e0616cf7206808f959409e83b66ea89d106d5856b6db
Input/saved output/bundle/manifest unchanged; fitted state: 7623ab0d4d555aed3c73944f59f85acb23e141ada2794fa826c9d7231383b7f7
Temporary CSV files removed; no labels, accuracy/F1 or model tuning.


### Интерпретация и самопроверка

Фактические ответы сохраняются: `01/04/05 → positive`, остальные → `neutral`. Negative отсутствует — coverage трёх классов не является требованием smoke demo. `02` с net loss даёт neutral, `04` с did not increase и `05` с less than last year — positive; эти inputs и frozen model не исправляются по ответам. По одному ответу не установлено, какие features вызвали предсказание. Примеры иллюстрируют ограничения финансового контекста, отрицаний и сравнений, но не измеряют частоту ошибок.

У `06` HTML tags удалены, `&amp;` декодировано, регистр нормализован; числа/валюта сохранены. OOV-only `08` даёт нулевой TF-IDF row и штатный neutral label: vocabulary не расширяется, confidence/abstention API не обещает. Fitted state, model pair и принятый demo output неизменны.

1. Почему demo без true labels не позволяет посчитать macro F1?
2. Почему отсутствие negative не требует добавить «удобную» фразу?
3. Что означает `(8, 5000)` и почему OOV не вызывает fit нового vocabulary?
4. Почему одинаковые inputs дают одинаковые ответы без изменения IDF/weights?
5. Какие поля CSV сохраняются и почему existing output нельзя заменять?
6. Почему проверка duplicates использует прежние demo inputs, а не качество модели?

Следующий пункт — D7-05: итоговый README. Training/CV/final evaluation и README здесь не запускались/не менялись.
